In [1]:
#Python data preparation for Power bi
import pandas as pd
import re


In [5]:
# 1. LOAD DATASET
# ============================================================

file_path = "/content/clean_jobs_descriptions_combined (1).csv"

df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


Dataset Shape: (8785, 13)

Columns:
['Job Id', 'Job Title', 'Company', 'location', 'Job Description', 'Experience', 'Qualifications', 'Salary Range', 'Work Type', 'skills', 'clean_description', 'extracted_skills', 'categories']


In [7]:
 #2. BASIC DATA CLEANING
# ============================================================

# Remove duplicate Job IDs
df = df.drop_duplicates(subset=["Job Id"])

# Fill missing NLP results
df["extracted_skills"] = df["extracted_skills"].fillna("")
df["categories"] = df["categories"].fillna("")

print("\nAfter removing duplicate jobs:")
print("Total Jobs:", df["Job Id"].nunique())


After removing duplicate jobs:
Total Jobs: 8685


In [8]:
# 3. CREATE JOB TABLE
# ============================================================

jobs_table = df[
    [
        "Job Id",
        "Job Title",
        "Company",
        "location",
        "Experience",
        "Work Type"
    ]
].copy()

print("\nJobs Table:")
print(jobs_table.shape)


Jobs Table:
(8685, 6)


In [11]:
# 4. CREATE SKILL ANALYTICS TABLE
# ============================================================

skill_rows = []

for _, row in df.iterrows():

    skills = row["extracted_skills"]

    if not skills:
        continue

    # Split multiple extracted skills
    skill_list = skills.split(",")

    for skill in skill_list:

        skill = skill.strip()

        if not skill:
            continue

        # Remove category written inside brackets
        # Example:
        # Python (Programming)
        # becomes:
        # Python

        skill_clean = re.sub(
            r"\s*\([^)]*\)",
            "",
            skill
        )

        skill_clean = skill_clean.strip()

        if skill_clean:

            skill_rows.append({
                "Job Id": row["Job Id"],
                "Job Title": row["Job Title"],
                "Skill": skill_clean
            })


skills_table = pd.DataFrame(skill_rows)


# Remove duplicate skill for same job
skills_table = skills_table.drop_duplicates(
    subset=["Job Id", "Skill"]
)


print("\nSkill Table:")
print(skills_table.shape)

print(skills_table.head(20))


Skill Table:
(3128, 3)
          Job Id               Job Title       Skill
0   3.984540e+14           Web Developer  JavaScript
1   3.984540e+14           Web Developer        HTML
2   3.984540e+14           Web Developer         CSS
5   3.984540e+14           Web Developer       React
6   3.984540e+14           Web Developer     Angular
7   2.808730e+15       Software Engineer      Python
8   2.808730e+15       Software Engineer        Java
9   2.808730e+15       Software Engineer  JavaScript
10  2.808730e+15       Software Engineer         SQL
12  1.355330e+15            UI Developer  JavaScript
13  1.355330e+15            UI Developer        HTML
14  1.355330e+15            UI Developer         CSS
17  9.818520e+14            Data Analyst         SQL
19  9.818520e+14            Data Analyst    Power BI
20  9.818520e+14            Data Analyst     Tableau
21  2.812080e+15         Software Tester      Python
22  2.812080e+15         Software Tester        Java
23  3.859840e+14  Data

In [10]:
# 5. TOP 20 SKILLS
# ============================================================

top_skills = (
    skills_table
    .groupby("Skill")["Job Id"]
    .nunique()
    .reset_index(name="Job Demand")
    .sort_values("Job Demand", ascending=False)
    .head(20)
)

print("\n==============================")
print("TOP 20 SKILLS")
print("==============================")

print(top_skills)


TOP 20 SKILLS
         Skill  Job Demand
18      Python         358
21         SQL         353
11  JavaScript         247
10        Java         242
8         HTML         198
3          CSS         198
2        Azure         130
0          AWS         130
24     Tableau         127
20       React         112
16    Power BI         108
1      Angular          90
5        Excel          87
15      Oracle          85
19           R          79
23       Spark          73
9       Hadoop          73
14       MySQL          67
7          Git          63
6          GCP          58


In [12]:
# 6. SKILL DEMAND BY JOB ROLE
# ============================================================

role_skill_demand = (
    skills_table
    .groupby(["Job Title", "Skill"])["Job Id"]
    .nunique()
    .reset_index(name="Job Demand")
    .sort_values("Job Demand", ascending=False)
)

print("\n==============================")
print("SKILL DEMAND BY JOB ROLE")
print("==============================")

print(role_skill_demand.head(20))



SKILL DEMAND BY JOB ROLE
                 Job Title       Skill  Job Demand
70       Software Engineer  JavaScript         120
72       Software Engineer      Python         112
74       Software Engineer         SQL          72
69       Software Engineer        Java          72
78   Systems Administrator      Oracle          67
79   Systems Administrator         SQL          67
77   Systems Administrator       MySQL          67
10            Data Analyst         SQL          58
31     Front-End Developer         CSS          54
32     Front-End Developer        HTML          54
40          Java Developer        Java          48
62       Software Engineer     Angular          48
67       Software Engineer         Git          48
73       Software Engineer       React          48
64       Software Engineer         CSS          48
68       Software Engineer        HTML          48
13            Data Analyst     Tableau          47
6             Data Analyst    Power BI          47
23  D

In [13]:
# 7. CREATE CATEGORY TABLE
# ============================================================

category_rows = []

for _, row in df.iterrows():

    categories = row["categories"]

    if not categories:
        continue

    category_list = categories.split(",")

    for category in category_list:

        category = category.strip()

        if category:

            category_rows.append({
                "Job Id": row["Job Id"],
                "Job Title": row["Job Title"],
                "Category": category
            })


categories_table = pd.DataFrame(category_rows)


# Remove duplicate job-category combinations
categories_table = categories_table.drop_duplicates(
    subset=["Job Id", "Category"]
)


print("\nCategory Table:")
print(categories_table.shape)

print(categories_table.head(10))


Category Table:
(2069, 3)
         Job Id          Job Title         Category
0  3.984540e+14      Web Developer      Programming
1  3.984540e+14      Web Developer  Web Development
2  2.808730e+15  Software Engineer      Programming
3  2.808730e+15  Software Engineer         Database
4  1.355330e+15       UI Developer      Programming
5  1.355330e+15       UI Developer  Web Development
6  9.818520e+14       Data Analyst      Programming
7  9.818520e+14       Data Analyst         Database
8  9.818520e+14       Data Analyst   Data Analytics
9  2.812080e+15    Software Tester      Programming


In [14]:
# 8. CATEGORY DISTRIBUTION
# ============================================================

category_distribution = (
    categories_table
    .groupby("Category")["Job Id"]
    .nunique()
    .reset_index(name="Job Demand")
    .sort_values("Job Demand", ascending=False)
)

print("\n==============================")
print("CATEGORY DISTRIBUTION")
print("==============================")

print(category_distribution)



CATEGORY DISTRIBUTION
           Category  Job Demand
6       Programming         869
3          Database         387
7   Web Development         241
2    Data Analytics         214
1             Cloud         130
4            DevOps         118
0          Big Data          73
5  Machine Learning          37


In [15]:
# 9. KPI — TOTAL JOBS
# ============================================================

total_jobs = df["Job Id"].nunique()


# ============================================================
# 10. KPI — UNIQUE SKILLS
# ============================================================

unique_skills = skills_table["Skill"].nunique()


# ============================================================
# 11. KPI — TOP SKILL
# ============================================================

if len(top_skills) > 0:

    top_skill = top_skills.iloc[0]["Skill"]
    top_skill_demand = top_skills.iloc[0]["Job Demand"]
else:

    top_skill = "N/A"
    top_skill_demand = 0

In [16]:
# 12. KPI — TOP JOB ROLE
# ============================================================

job_role_demand = (
    df.groupby("Job Title")["Job Id"]
    .nunique()
    .reset_index(name="Job Demand")
    .sort_values("Job Demand", ascending=False)
)

if len(job_role_demand) > 0:

    top_job_role = job_role_demand.iloc[0]["Job Title"]

else:

    top_job_role = "N/A"


In [17]:
# 13. KPI — TOP SKILL CATEGORY
# ============================================================

if len(category_distribution) > 0:

    top_category = category_distribution.iloc[0]["Category"]
    top_category_demand = category_distribution.iloc[0]["Job Demand"]

else:

    top_category = "N/A"
    top_category_demand = 0

In [18]:
# 14. KPI SUMMARY
# ============================================================

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Jobs",
        "Unique Skills",
        "Top Skill",
        "Top Skill Demand",
        "Top Job Role",
        "Top Skill Category",
        "Top Category Demand"
    ],

    "Value": [
        total_jobs,
        unique_skills,
        top_skill,
        top_skill_demand,
        top_job_role,
        top_category,
        top_category_demand
    ]
})


print("\n==============================")
print("KPI SUMMARY")
print("==============================")

print(kpi_summary)


# ============================================================
# 15. SAVE DATA FOR POWER BI
# ============================================================

jobs_table.to_csv(
    "PowerBI_Jobs.csv",
    index=False
)

skills_table.to_csv(
    "PowerBI_Skills.csv",
    index=False
)

categories_table.to_csv(
    "PowerBI_Categories.csv",
    index=False
)

top_skills.to_csv(
    "PowerBI_Top20_Skills.csv",
    index=False
)

role_skill_demand.to_csv(
    "PowerBI_Role_Skill_Demand.csv",
    index=False
)

category_distribution.to_csv(
    "PowerBI_Category_Distribution.csv",
    index=False
)

kpi_summary.to_csv(
    "PowerBI_KPI_Summary.csv",
    index=False
)


# ============================================================
# 16. FINAL OUTPUT
# ============================================================

print("\n==========================================")
print("DAY 18 PYTHON PROCESSING COMPLETED")
print("==========================================")

print("\nFiles created:")

print("1. PowerBI_Jobs.csv")
print("2. PowerBI_Skills.csv")
print("3. PowerBI_Categories.csv")
print("4. PowerBI_Top20_Skills.csv")
print("5. PowerBI_Role_Skill_Demand.csv")
print("6. PowerBI_Category_Distribution.csv")
print("7. PowerBI_KPI_Summary.csv")

print("\nTotal Jobs:", total_jobs)
print("Unique Skills:", unique_skills)
print("Top Skill:", top_skill)
print("Top Job Role:", top_job_role)
print("Top Skill Category:", top_category)


KPI SUMMARY
                   KPI           Value
0           Total Jobs            8685
1        Unique Skills              26
2            Top Skill          Python
3     Top Skill Demand             358
4         Top Job Role  UX/UI Designer
5   Top Skill Category     Programming
6  Top Category Demand             869

DAY 18 PYTHON PROCESSING COMPLETED

Files created:
1. PowerBI_Jobs.csv
2. PowerBI_Skills.csv
3. PowerBI_Categories.csv
4. PowerBI_Top20_Skills.csv
5. PowerBI_Role_Skill_Demand.csv
6. PowerBI_Category_Distribution.csv
7. PowerBI_KPI_Summary.csv

Total Jobs: 8685
Unique Skills: 26
Top Skill: Python
Top Job Role: UX/UI Designer
Top Skill Category: Programming
